In [ ]:
import utils
import os
import pickle
import algo
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import copy
import seaborn as sns
%matplotlib widget

In [ ]:
def prepare_data_subj(Subj_ID, fs):
    eeg_list, eog_list, gaze_list, feats_list = utils.load_subj(Subj_ID)
    gaze_coords_list = [gaze[:,0:2,:] for gaze in gaze_list]
    saccade_list = [np.expand_dims(gaze[:,2,:], axis=1) for gaze in gaze_list]
    blink_list = [np.expand_dims(gaze[:,3,:], axis=1) for gaze in gaze_list]
    saccade_list = utils.refine_saccades(saccade_list, blink_list)
    gaze_velocity_list = [utils.calcu_gaze_velocity(gaze) for gaze in gaze_list]
    objflow_list = [np.expand_dims(feats[:,8,:], axis=1) for feats in feats_list]
    eeg_reg_list = [utils.regress_out(eeg, eog) for eeg, eog in zip(eeg_list, eog_list)]
    eeg_list = [utils.remove_shot_cuts_and_center(d, fs, remove_time=1) for d in eeg_list]
    eog_list = [utils.remove_shot_cuts_and_center(d, fs, remove_time=1) for d in eog_list]
    gaze_coords_list = [utils.remove_shot_cuts_and_center(d, fs, remove_time=1) for d in gaze_coords_list]
    saccade_list = [utils.remove_shot_cuts_and_center(d, fs, remove_time=1, CENTER=False) for d in saccade_list]
    blink_list = [utils.remove_shot_cuts_and_center(d, fs, remove_time=1, CENTER=False) for d in blink_list]
    gaze_velocity_list = [utils.remove_shot_cuts_and_center(d, fs, remove_time=1) for d in gaze_velocity_list]
    objflow_list = [utils.remove_shot_cuts_and_center(d, fs, remove_time=1) for d in objflow_list]
    eeg_reg_list = [utils.remove_shot_cuts_and_center(d, fs, remove_time=1) for d in eeg_reg_list]
    data_multitask_dict = {'EEG': eeg_list, 'EOG': eog_list, 'GAZE': gaze_coords_list, 'GAZE_V': gaze_velocity_list, 'EEG-EOG': eeg_reg_list}
    return data_multitask_dict, objflow_list, saccade_list, blink_list

In [ ]:
def mask_data(data_multitask_dict, saccade_list, blink_list, objflow_list, nb_nearby_samples=[9,3], MOD=None, MASK_EVENTS=False):
    data_masked_dict = copy.deepcopy(data_multitask_dict)
    objflow_masked_list = copy.deepcopy(objflow_list)
    mask_list = [utils.get_mask_from_gaze(xy_multitask, saccade_multitask, blink_multitask, nb_nearby_samples=nb_nearby_samples) for xy_multitask, saccade_multitask, blink_multitask in zip(data_multitask_dict['GAZE'], saccade_list, blink_list)]
    if MASK_EVENTS:
        event_mask_list = utils.create_event_masks(False, event_type='circle', SURROUND=False)
        event_mask_list = [np.expand_dims(mask, axis=1) for mask in event_mask_list]
        mask_list = [mask | ~event_mask for mask, event_mask in zip(mask_list, event_mask_list)]
    MOD_list = [MOD] if MOD is not None else data_multitask_dict.keys()
    for mod in MOD_list:
        data_dim = data_multitask_dict[mod][0].shape[1]
        mask_list_mod = [np.repeat(mask, data_dim, axis=1) for mask in mask_list]
        for data, mask in zip(data_masked_dict[mod], mask_list_mod):
            data[mask] = np.nan
    feats_dim = objflow_list[0].shape[1]
    mask_list_mod = [np.repeat(mask, feats_dim, axis=1) for mask in mask_list]
    for feats, mask in zip(objflow_masked_list, mask_list_mod):
        feats[mask] = np.nan
    return data_masked_dict, objflow_masked_list, mask_list

## Load data

Subject 12 was excluded from the analysis because of a technical error in the eye-tracker

In [ ]:
subjects = ['Subj_1', 'Subj_2', 'Subj_3', 'Subj_4', 'Subj_5', 'Subj_6', 'Subj_7', 'Subj_8', 'Subj_9', 'Subj_10', 'Subj_11', 'Subj_13', 'Subj_14', 'Subj_15']
subj_path = [f"C:/Users/yyao/Documents/Experiments/data/SOMove_MultiTask/{sub}/" for sub in subjects]
bads = [[], ['A6','A24','B14','B25'], ['B25','B26'], ['A2','A8','A9','A15','B25','B26','B27','B31'], ['B25','B31'], ['B12'], ['B19','A30'], ['A5','A9','B25'], ['B25'], ['B15'], ['A20','A30','A31','B25'], ['B25'], ['B31'], ['B31']] # bad channels
fs = 30
feats_path_folder = '../Feat_Multi/features/'
MODs = ['EEG', 'EOG', 'GAZE', 'GAZE_V', 'EEG-EOG']
# create table and figure folders for each modality
for mod in MODs:
    table_folder = 'tables/' + mod + '/'
    fig_folder = 'figures/' + mod + '/'
    if not os.path.exists(table_folder):
        os.makedirs(table_folder)
    if not os.path.exists(fig_folder):
        os.makedirs(fig_folder)

In [ ]:
%%capture
LOAD_ONLY = True
ALL_NEW = False
len_video = 180
if not LOAD_ONLY:
    for TASK in ['1', '2', '3']:
        if ALL_NEW:
            _, _, _, _, _ = utils.data_multi_subj(subj_path, fs, bads, feats_path_folder, len_video, SAVE=True, Task=TASK)
        else:
            _, _, _, _, _ = utils.add_new_data(subj_path, fs, bads, feats_path_folder, len_video, Task=TASK)

In [ ]:
for subj_ID in range(len(subjects)):
    for task_ID in range(1, 4):
        data_multitask_dict, objflow_list, saccade_list, blink_list = prepare_data_subj(subj_ID, fs)
        utils.check_alignment(subj_ID+1, task_ID, data_multitask_dict['EOG'], data_multitask_dict['GAZE'], nb_points=1000)

In [ ]:
def acc_fine_grained(corr_match_eeg, corr_mismatch_eeg, nb_blocks, nb_folds=7, range_into_account=3, nb_comp_into_account=2):
    corr_m_folds = np.array_split(corr_match_eeg, nb_folds, axis=0)
    corr_mm_folds = np.array_split(corr_mismatch_eeg, nb_folds, axis=0)
    corr_m_blocks = [np.array_split(corr_m, nb_blocks, axis=0) for corr_m in corr_m_folds]
    corr_mm_blocks = [np.array_split(corr_mm, nb_blocks, axis=0) for corr_mm in corr_mm_folds]
    block_res_folds = []
    for i in range(nb_folds):
        block_res = [utils.eval_compete_3D(m, mm, True, range_into_account=range_into_account, nb_comp_into_account=nb_comp_into_account, message=False)[0] for m, mm in zip(corr_m_blocks[i], corr_mm_blocks[i])]
        block_res = np.stack(block_res, axis=0)
        block_res_folds.append(block_res)
    block_res_avg = np.mean(block_res_folds, axis=0)
    return block_res_avg

In [ ]:
def process_corr(corr_tensor, range_into_account=3, nb_comp_into_account=2):
    _, _, nb_tasks = corr_tensor.shape
    corr_tensor = corr_tensor[:,:range_into_account,:]
    for i in range(nb_tasks):
        # sort the components for each task from the most to the least correlated
        corr_tensor[:, :, i] = np.sort(corr_tensor[:, :, i], axis=1)[:, ::-1]
    # take the sum of the first nb_comp_into_account components
    corr_tensor = np.sum(corr_tensor[:, :nb_comp_into_account, :], axis=1) # shape: (nb_trials, nb_tasks)
    return corr_tensor


## Gaze data statistics

In [ ]:
dva_records = []
for subj_ID in range(len(subjects)):
    data_multitask_dict, _, _, _ = prepare_data_subj(subj_ID, fs)
    gaze_all_videos_tasks = data_multitask_dict['GAZE']
    for video_idx, gaze_video in enumerate(gaze_all_videos_tasks):
        for task_ID in range(1, 4):
            xy = gaze_video[:, :, task_ID-1]
            _, dva, _, _ = utils.fixation_cluster(xy, eps=10)
            dva_records.append({
                'Subject_ID': subj_ID,
                'Video_ID': video_idx,
                'Task_ID': task_ID,
                'DVA': dva
            })
df_dva = pd.DataFrame(dva_records)

In [ ]:
# Group the data by Task_ID and calculate both mean and standard deviation
task_stats = df_dva.groupby('Task_ID').agg(
    Average_DVA=('DVA', 'mean'),
    Std_DVA=('DVA', 'std')
).reset_index()

print("DVA Statistics per Task:")
print(task_stats)

## CCA-Block Analysis

This part is to check if the strength of the neural coupling is a function of the eccentricity of the stimulus. However, the results were not conclusive, and therefore this part of the analysis was not included in the final manuscript. Jump to the next section for the main analysis.

In [ ]:
L_EEG = 3
L_Stim = 15
offset_EEG = 1
offset_Stim = 0

MOD = 'EEG-EOG'
trial_len = 45
task_train = [1, 2, 3]
nb_nearby_samples = None # [9, 3]
BOOTSTRAP = True
MASK = True
n_components = 5 if (MOD != 'GAZE_V' and MOD != 'GAZE') else 3
range_into_account = 3
nb_comp_into_account = 2

In [ ]:
def block_file_name(kind, save_name, block_len, block_ol, trial_len, task_train, BOOTSTRAP, MASK, nb_nearby_samples, nb_mismatch, mismatch_scope, MOD):
    """File name of a block analysis whose mismatch segments are not taken from the block itself."""
    return (f"tables/{MOD}/{save_name}_{kind}_blocklen{block_len}_blockol{block_ol}_triallen{trial_len}_train_{task_train}"
            f"{'_BT' if BOOTSTRAP else ''}{'_masked' if MASK else ''}"
            f"{('_nearby' + str(nb_nearby_samples)) if nb_nearby_samples is not None else ''}"
            f"{('_mismatch' + str(nb_mismatch)) if nb_mismatch is not None else ''}"
            f"_mm{mismatch_scope}.pickle")


def analyze_all_blocks_global(block_len, block_ol, fs, L_EEG, L_Stim, offset_EEG, offset_Stim, task_train=[2,3], trial_len=30, n_components=5, save_name=None, MOD='EEG-EOG', BOOTSTRAP=True, MASK=False, nb_nearby_samples=None, nb_mismatch=None, mismatch_scope='other_videos', guard_len=None):
    """Same as analyze_all_blocks, except that the mismatch segments are not drawn from the block
    the matched segment comes from. Since trial_len is close to block_len, and masking shortens the
    block further, a within-block competitor is largely a shifted copy of the matched segment, and
    raising nb_mismatch does not make the competitors any more diverse.
    mismatch_scope: 'other_videos' to draw the competitors from the videos that are not tested in
    the current fold, 'test_video' to draw them from the test video, at least guard_len s
    (trial_len by default, i.e. no overlap) away from the matched segment. Note that in the latter
    case the matched segments for which the test video is too short to provide such a competitor
    are discarded. The matched segments, and hence the block they are attached to, are unchanged."""
    corr_match_dict_all = {}
    corr_mismatch_dict_all = {}
    rts_kept_dict_all = {}
    for Subj_ID in range(len(subjects)):
        print(f"###################\nSubject {Subj_ID + 1} / {len(subjects)}")
        data_multitask_dict, objflow_list, saccade_list, blink_list = prepare_data_subj(Subj_ID, fs)
        if MASK:
            data_masked_dict, objflow_masked_list, _ = mask_data(data_multitask_dict, saccade_list, blink_list, objflow_list, nb_nearby_samples, MOD=MOD)
            data_masked_list = data_masked_dict[MOD]
        else:
            data_masked_list = None
            objflow_masked_list = None
        CCA = algo.CanonicalCorrelationAnalysis(data_multitask_dict[MOD], objflow_list, fs, L_EEG, L_Stim, offset_EEG, offset_Stim, task_train=task_train, leave_out=1, n_components=n_components, EEG_masked=data_masked_list, Stim_masked=objflow_masked_list)
        nb_compete = nb_mismatch if nb_mismatch is not None else 1

        corr_match_dict, corr_mismatch_dict, rts_kept_dict = CCA.mm_blocks_global_mismatch(trial_len, BOOTSTRAP=BOOTSTRAP, overlap=0.9, block_len=block_len, block_ol=block_ol, nb_compete=nb_compete, mismatch_scope=mismatch_scope, guard_len=guard_len)
        corr_match_dict_all[Subj_ID] = corr_match_dict
        corr_mismatch_dict_all[Subj_ID] = corr_mismatch_dict
        rts_kept_dict_all[Subj_ID] = rts_kept_dict

    if save_name is not None:
        results = {'match': corr_match_dict_all, 'mismatch': corr_mismatch_dict_all, 'rts': rts_kept_dict_all}
        for kind, res in results.items():
            with open(block_file_name(kind, save_name, block_len, block_ol, trial_len, task_train, BOOTSTRAP, MASK, nb_nearby_samples, nb_mismatch, mismatch_scope, MOD), 'wb') as f:
                pickle.dump(res, f)
    return corr_match_dict_all, corr_mismatch_dict_all, rts_kept_dict_all

In [ ]:
block_len = 75
block_ol = 0.8
nb_mismatch = 10

In [ ]:
mismatch_scope = 'other_videos'
RUN_GLOBAL_MM = False  # set to False to load the results of a previous run

if RUN_GLOBAL_MM:
    corr_match_dict_all, corr_mismatch_dict_all, rts_kept_dict_all = analyze_all_blocks_global(block_len, block_ol, fs, L_EEG, L_Stim, offset_EEG, offset_Stim, task_train=task_train, trial_len=trial_len, n_components=n_components, save_name="RUN_1", MOD=MOD, BOOTSTRAP=BOOTSTRAP, MASK=MASK, nb_nearby_samples=nb_nearby_samples, nb_mismatch=nb_mismatch, mismatch_scope=mismatch_scope)
else:
    for kind, var in [('match', 'corr_match_dict_all'), ('mismatch', 'corr_mismatch_dict_all'), ('rts', 'rts_kept_dict_all')]:
        with open(block_file_name(kind, "RUN_1", block_len, block_ol, trial_len, task_train, BOOTSTRAP, MASK, nb_nearby_samples, nb_mismatch, mismatch_scope, MOD), 'rb') as f:
            globals()[var] = pickle.load(f)

In [ ]:
MIN_FOLDS = 4      # a subject x block accuracy has to rest on at least this many videos (out of 7)
MIN_SUBJECTS = 10  # a block has to keep at least this many subjects after applying MIN_FOLDS

nb_compete_used = nb_mismatch if nb_mismatch is not None else 1
all_blocks = sorted({b for s in corr_match_dict_all for b in corr_match_dict_all[s]})


def drop_cell(subj_id, idx_block):
    corr_match_dict_all[subj_id].pop(idx_block, None)
    corr_mismatch_dict_all[subj_id].pop(idx_block, None)
    rts_kept_dict_all[subj_id].pop(idx_block, None)


nb_videos_cell = {}
print(f"{'block':>5} {'subj':>5} {'videos':>7} {'trials':>7} {'decisions':>10}   videos per subject")
for idx_block in all_blocks:
    subj_in_block = [s for s in corr_match_dict_all if idx_block in corr_match_dict_all[s]]
    nb_videos_cell[idx_block] = {}
    for subj_id in subj_in_block:
        rts = rts_kept_dict_all[subj_id][idx_block]
        assert rts is not None, "The number of videos per cell is read from rts_kept_dict_all, which is only filled in when MASK=True."
        nb_videos_cell[idx_block][subj_id] = rts.shape[0]
    nb_trials = sum(corr_match_dict_all[s][idx_block].shape[0] for s in subj_in_block)//nb_compete_used
    print(f"{idx_block:5d} {len(subj_in_block):5d} {sum(nb_videos_cell[idx_block].values()):7d} {nb_trials:7d} "
          f"{nb_trials*nb_compete_used:10d}   " + ' '.join(f'S{s+1}:{n}' for s, n in sorted(nb_videos_cell[idx_block].items())))

# a subject x block accuracy that rests on too few videos is both noisy and tied to whichever
# videos happened to survive the mask
for idx_block in all_blocks:
    for subj_id, nb_videos in sorted(nb_videos_cell[idx_block].items()):
        if nb_videos < MIN_FOLDS:
            drop_cell(subj_id, idx_block)
            print(f'dropped subject {subj_id + 1} from block {idx_block}: {nb_videos} video(s) < MIN_FOLDS')

# a block that is left with too few subjects is not comparable to the others
for idx_block in all_blocks:
    subj_left = [s for s in corr_match_dict_all if idx_block in corr_match_dict_all[s]]
    if len(subj_left) < MIN_SUBJECTS:
        for subj_id in subj_left:
            drop_cell(subj_id, idx_block)
        print(f'dropped block {idx_block}: only {len(subj_left)} subject(s) left < MIN_SUBJECTS')

blocks_left = sorted({b for s in corr_match_dict_all for b in corr_match_dict_all[s]})
print('\nblocks left:', blocks_left)
print('subjects per block:', {b: sum(1 for s in corr_match_dict_all if b in corr_match_dict_all[s]) for b in blocks_left})

In [ ]:
# Dynamically find all unique block IDs across ALL subjects
all_blocks = set()
for subj_id in corr_match_dict_all.keys():
    all_blocks.update(corr_match_dict_all[subj_id].keys())
sorted_blocks = sorted(list(all_blocks))  # Sorted so blocks process in order (e.g., 1, 2, 3...)

# Dictionaries to store your results
acc_per_block_per_subject = {}
subjects_per_block = {}  # Tracks which subjects are in which block matrix

# Iterate through every unique block found
for idx_block in sorted_blocks:
    print(f"\n--- Processing Block: {idx_block} ---")
    block_subj_accs = []
    subjects_included = []
    
    # Check every subject to see if they have this specific block
    for subj_id in corr_match_dict_all.keys():
        if idx_block in corr_match_dict_all[subj_id]:
            match_data = corr_match_dict_all[subj_id][idx_block]
            mismatch_data = corr_mismatch_dict_all[subj_id][idx_block]
            
            # Evaluate accuracy for this subject's block
            acc_all_tasks, _, _, _, _ = utils.eval_compete_3D(
                match_data, 
                mismatch_data, 
                True, 
                range_into_account=range_into_account, 
                nb_comp_into_account=nb_comp_into_account, 
                message=False
            )
            
            block_subj_accs.append(acc_all_tasks)
            subjects_included.append(subj_id)
            
    # Only create a matrix if at least one subject had this block
    if block_subj_accs:
        acc_per_block_per_subject[idx_block] = np.stack(block_subj_accs, axis=0)
        subjects_per_block[idx_block] = subjects_included
        
        print(f"Block {idx_block} matrix shape: {acc_per_block_per_subject[idx_block].shape}")
        print(f"Subjects in this block: {subjects_included}")

In [ ]:
# 1. Convert your dictionary data into a "tidy" long-form DataFrame
plot_data = []

for block_id, acc_matrix in acc_per_block_per_subject.items():
    # acc_matrix shape: (nb_subjects, nb_tasks)
    subjects_in_block = subjects_per_block[block_id]
    
    for subj_idx, subj_id in enumerate(subjects_in_block):
        for task_idx in range(acc_matrix.shape[1]):
            plot_data.append({
                'Block': block_id,
                'Subject': subj_id,
                'Task': f'Task {task_idx + 1}',  # Change this if you have a list of task names
                'Accuracy': acc_matrix[subj_idx, task_idx]
            })

df_plot = pd.DataFrame(plot_data)

# Ensure blocks are sorted chronologically/numerically on the X-axis
try:
    df_plot['Block'] = pd.to_numeric(df_plot['Block'])
    df_plot = df_plot.sort_values('Block')
except ValueError:
    df_plot = df_plot.sort_values('Block')

# Filter to keep only Task 2 and Task 3
# df_filtered = df_plot[df_plot['Task'].isin(['Task 2', 'Task 3'])]
df_filtered = df_plot

# Define your new desired order
task_order = ['Task 1', 'Task 2', 'Task 3']

In [ ]:
plt.figure(figsize=(8, 5))

# Grouped boxplots using filtered data
sns.boxplot(
    data=df_filtered,             # <-- Use the filtered data
    x='Block', 
    y='Accuracy', 
    hue='Task', 
    hue_order=task_order,         # <-- Updated order
    palette='Set2', 
    width=0.5                     # Slightly narrower width since there are fewer boxes
)

# Overlay individual subject data points
sns.stripplot(
    data=df_filtered,             # <-- Use the filtered data
    x='Block', 
    y='Accuracy', 
    hue='Task', 
    hue_order=task_order, 
    dodge=True, 
    color='black', 
    alpha=0.2, 
    legend=False
)

plt.title('Accuracy for 3 Tasks across All Blocks')
plt.xlabel('Window Index')
plt.ylabel('Accuracy')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.legend(title='Tasks')

plt.tight_layout()
plt.savefig('acc_vs_blocks_75_80.png', dpi=300)
plt.show()

## CCA

In [ ]:
L_EEG = 3
L_Stim = 15
offset_EEG = 1
offset_Stim = 0

RUN_TIMES = 5
MOD = 'EEG-EOG'
trial_len = 45
task_train = [1, 2, 3]
nb_nearby_samples = None # note that nearby samples have already been removed due to hankelization
BOOTSTRAP = True
MASK = True
MASK_EVENTS = True # if True, mask cross and circle events as well
if MASK_EVENTS:
    assert MASK, "MASK_EVENTS is set to True, but MASK is False. Please set MASK to True to mask events."
n_components = 5 if (MOD != 'GAZE_V' and MOD != 'GAZE') else 3
range_into_account = 3
nb_comp_into_account = 2

In [ ]:
def analyze_all(fs, L_EEG, L_Stim, offset_EEG, offset_Stim, range_into_account, nb_comp_into_account, task_train=[2,3], trial_len=30, n_components=5, save_name=None, MOD='EEG-EOG', PERMU_TEST=False, BOOTSTRAP=True, MASK=False, MASK_EVENTS=False, nb_nearby_samples=None, nb_mismatch=None):
    all_acc = []
    corr_match_all_subj = []
    corr_mismatch_all_subj = []
    rts_kept_all_subj = []
    acc_permu_all = []
    start_points = None
    for Subj_ID in range(len(subjects)):
        print(f"###################\nSubject {Subj_ID + 1} / {len(subjects)}")
        data_multitask_dict, objflow_list, saccade_list, blink_list = prepare_data_subj(Subj_ID, fs)
        if MASK:
            data_masked_dict, objflow_masked_list, _ = mask_data(data_multitask_dict, saccade_list, blink_list, objflow_list, nb_nearby_samples, MOD=MOD, MASK_EVENTS=MASK_EVENTS)
            data_masked_list = data_masked_dict[MOD]
        else:
            data_masked_list = None
            objflow_masked_list = None
        CCA = algo.CanonicalCorrelationAnalysis(data_multitask_dict[MOD], objflow_list, fs, L_EEG, L_Stim, offset_EEG, offset_Stim, task_train=task_train, leave_out=1, n_components=n_components, EEG_masked=data_masked_list, Stim_masked=objflow_masked_list)
        nb_compete = nb_mismatch if nb_mismatch is not None else 1
        corr_match_data, corr_mismatch_data, acc_permu_list, start_points, rts_kept = CCA.match_mismatch(trial_len=trial_len, PERMU_TEST=PERMU_TEST, BOOTSTRAP=BOOTSTRAP, given_start_points=start_points, nb_compete=nb_compete)
        print('Average ratio of kept data: {}'.format(np.mean(rts_kept, axis=0)*100))
        print("###########Match-Mismatch, TASK 1, 2, 3###########")
        acc_all_tasks, _, _, _, _ = utils.eval_compete_3D(corr_match_data, corr_mismatch_data, True, range_into_account=range_into_account, nb_comp_into_account=nb_comp_into_account, message=True)
        if not MASK:
            print("###########TASK 2 vs TASK 1###########")
            acc_2vs1, _, _, _, _ = utils.eval_compete(corr_match_data[:,:,1], corr_match_data[:,:,0], True, range_into_account=range_into_account, nb_comp_into_account=nb_comp_into_account, message=True)
            print("###########TASK 3 vs TASK 2###########")
            acc_3vs2, _, _, _, _ = utils.eval_compete(corr_match_data[:,:,2], corr_match_data[:,:,1], True, range_into_account=range_into_account, nb_comp_into_account=nb_comp_into_account, message=True)
            all_acc.append({'Subject': Subj_ID + 1, 'Task_1': acc_all_tasks[0], 'Task_2': acc_all_tasks[1], 'Task_3': acc_all_tasks[2], 'Task_2vs1': acc_2vs1, 'Task_3vs2': acc_3vs2})
        else:
            all_acc.append({'Subject': Subj_ID + 1, 'Task_1': acc_all_tasks[0], 'Task_2': acc_all_tasks[1], 'Task_3': acc_all_tasks[2]})
        corr_match_all_subj.append(corr_match_data)
        corr_mismatch_all_subj.append(corr_mismatch_data)
        rts_kept_all_subj.append(rts_kept)
        if PERMU_TEST:
            acc_permu_all += acc_permu_list
    all_acc = pd.DataFrame(all_acc)
    if PERMU_TEST:
        acc_permu = np.concatenate(acc_permu_all, axis=0)
        alpha = 0.05
        lower_bound = np.percentile(acc_permu, alpha/2*100)
        upper_bound = np.percentile(acc_permu, (1-alpha/2)*100)
    else:
        lower_bound = None
        upper_bound = None
    # add two columns to all_acc for lower and upper bound
    all_acc['lower_bound'] = lower_bound
    all_acc['upper_bound'] = upper_bound
    if save_name is not None:
        save_path = f"tables/{MOD}/{save_name}"
        all_acc.to_csv(f"{save_path}_acc_{trial_len}_train_{task_train}{'_BT' if BOOTSTRAP else ''}{'_masked' if MASK else ''}{'_cirincluded' if MASK_EVENTS else ''}{('_nearby'+str(nb_nearby_samples)) if nb_nearby_samples is not None else ''}{('_nbmm'+str(nb_mismatch)) if nb_mismatch is not None else ''}.csv", index=False)
        # save corr_match_all_subj, corr_mismatch_all_subj, start_idx as dictionary
        corr_res = {
            'corr_match_all_subj': corr_match_all_subj,
            'corr_mismatch_all_subj': corr_mismatch_all_subj,
            'start_points': start_points,
            'rts_kept_all_subj': rts_kept_all_subj
        }
        # save as pickle file
        with open(f"{save_path}_corr_{trial_len}_train_{task_train}{'_BT' if BOOTSTRAP else ''}{'_masked' if MASK else ''}{'_cirincluded' if MASK_EVENTS else ''}{('_nearby'+str(nb_nearby_samples)) if nb_nearby_samples is not None else ''}{('_nbmm'+str(nb_mismatch)) if nb_mismatch is not None else ''}.pickle", 'wb') as f:
            pickle.dump(corr_res, f)
    return all_acc, corr_match_all_subj, corr_mismatch_all_subj, rts_kept_all_subj, start_points

In [ ]:
# create a dictionary to store the results
for i in range(RUN_TIMES):
    save_name = f"RUN_{i+1}"
    PERMU_TEST = (i == 0) 
    all_acc, corr_match_all_subj, corr_mismatch_all_subj, rts_kept_all_subj, start_idx = analyze_all(fs, L_EEG, L_Stim, offset_EEG, offset_Stim, range_into_account, nb_comp_into_account, n_components=n_components, save_name=save_name, MOD=MOD, task_train=task_train, trial_len=trial_len, PERMU_TEST=PERMU_TEST, BOOTSTRAP=BOOTSTRAP, MASK=MASK, MASK_EVENTS=MASK_EVENTS, nb_nearby_samples=nb_nearby_samples)
    print(all_acc)

In [ ]:
# load the results
all_acc = []
for i in range(RUN_TIMES):
    save_name = f"RUN_{i+1}"
    save_path = f"tables/{MOD}/{save_name}"
    all_acc.append(pd.read_csv(f"{save_path}_acc_{trial_len}_train_{task_train}{'_BT' if BOOTSTRAP else ''}{'_masked' if MASK else ''}{'_cirincluded' if MASK_EVENTS else ''}{('_nearby'+str(nb_nearby_samples)) if nb_nearby_samples is not None else ''}.csv"))
# average the results
all_acc = pd.concat(all_acc, ignore_index=True)
all_acc = all_acc.groupby(['Subject']).mean().reset_index()

In [ ]:
plt.figure(figsize=(5, 4))

# Create colormap for different subjects
n_subjects = len(all_acc)
colors = plt.cm.rainbow(np.linspace(0, 1, n_subjects))

# Plot individual subject lines with different colors
for idx, (row, color) in enumerate(zip(all_acc.iterrows(), colors)):
    plt.plot([1, 2, 3], 
            [row[1]['Task_1'], row[1]['Task_2'], row[1]['Task_3']], 
            'o-', alpha=0.7, color=color, label=f'Subject {int(row[1]["Subject"])}')
# Add horizontal line at significance levels (0.54 and 0.45)
plt.axhline(y=all_acc['upper_bound'].mean(), color='grey', linestyle='--', alpha=0.8)
plt.axhline(y=all_acc['lower_bound'].mean(), color='grey', linestyle='--', alpha=0.8)
# Customize plot
plt.xticks([1, 2, 3], 
          ['Task_1', 'Task_2', 'Task_3'], 
          rotation=45)
plt.ylabel('Accuracy')
plt.grid(True, alpha=0.3)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import false_discovery_control
stat_1_2 = utils.wilcoxon_effect(all_acc['Task_2'], all_acc['Task_1'], zero_method="wilcox", alternative="greater")
stat_2_3 = utils.wilcoxon_effect(all_acc['Task_3'], all_acc['Task_2'], zero_method="wilcox", alternative="greater")
print(f"Effect size (rank-biserial r) for Task 2 > Task 1: {stat_1_2['rank_biserial']}, Effect size for Task 3 > Task 2: {stat_2_3['rank_biserial']}")
print(f'Z value for Task 2 > Task 1: {stat_1_2["z"]}, Z value for Task 3 > Task 2: {stat_2_3["z"]}')
print(f'(Uncorrected) P-value for Task 2 > Task 1: {stat_1_2["p"]}, P-value for Task 3 > Task 2: {stat_2_3["p"]}')

In [ ]:
# 1. Define the specific labels with newline characters (\n) to split them
label_map = {
    'Task_1': 'Task 1\n(ignore, eccentric)',
    'Task_2': 'Task 2\n(attend, eccentric)',
    'Task_3': 'Task 3\n(attend, central)'
}

# 2. Prepare the Long Format Data (for Swarmplot)
melted_data = pd.melt(all_acc[['Task_1', 'Task_2', 'Task_3']].reset_index(), 
                      id_vars=['index'], var_name='Task', value_name='Accuracy')
melted_data['Subject'] = melted_data['index']
# Map the Task column to the new descriptive labels
melted_data['Task'] = melted_data['Task'].map(label_map)

# 3. Prepare the Wide Format Data (for Boxplot)
# Rename columns directly using the same map so x-axes match perfectly
_display_df = all_acc[['Task_1', 'Task_2', 'Task_3']].rename(columns=label_map)

# --- Plotting ---
plt.figure(figsize=(3.5, 4)) # Increased height slightly to accommodate 2-line labels

# Create box plot with new labels
sns.boxplot(data=_display_df, fill=False, width=0.5, color='black')

# Add individual points
sns.swarmplot(data=melted_data, x='Task', y='Accuracy', hue='Subject', 
              palette='Set1', alpha=0.8, size=4, legend=False)

# Add horizontal line at significance levels
plt.axhline(y=all_acc['upper_bound'].mean(), color='grey', linestyle='--', alpha=0.8)
# plt.axhline(y=all_acc['lower_bound'].mean(), color='grey', linestyle='--', alpha=0.8)

# --- Significance Brackets ---
y_max = all_acc[['Task_1', 'Task_2', 'Task_3']].max().max()
y_range = all_acc[['Task_1', 'Task_2', 'Task_3']].max().max() - all_acc[['Task_1', 'Task_2', 'Task_3']].min().min()

# Bracket for Task 1 vs Task 2 (Indices 0 and 1)
if stat_1_2["p"] < 0.05:
    bracket_height_12 = y_max + 0.05 * y_range
    plt.plot([0, 0, 1, 1], [bracket_height_12 - 0.01 * y_range, bracket_height_12, 
                            bracket_height_12, bracket_height_12 - 0.01 * y_range], 
             'k-', linewidth=1)
    plt.text(0.5, bracket_height_12 + 0.01 * y_range, '*', 
             ha='center', va='bottom')

# Bracket for Task 2 vs Task 3 (Indices 1 and 2)
if stat_2_3["p"] < 0.05:
    bracket_height_23 = y_max + 0.1 * y_range 
    plt.plot([1, 1, 2, 2], [bracket_height_23 - 0.01 * y_range, bracket_height_23, 
                            bracket_height_23, bracket_height_23 - 0.01 * y_range], 
             'k-', linewidth=1)
    plt.text(1.5, bracket_height_23 + 0.01 * y_range, '*', 
             ha='center', va='bottom')

# --- Final Formatting ---
sns.despine()  
plt.ylabel('Accuracy', fontsize=10)
plt.ylim(0.3, 0.9)
plt.xlabel('', fontsize=10)

# Changed rotation to 0 because split lines usually look better horizontal.
# If they still overlap, change back to 45 or 90.
plt.xticks(rotation=20, fontsize=9) 

plt.tight_layout()
# plt.savefig('../../Manuscript/Spatial_Bias/acc_tasks.pdf', dpi=600, bbox_inches='tight', pad_inches=0.01)
plt.show()

### Topoplot

In [ ]:
def topoplot_cca(fs, L_EEG, L_Stim, offset_EEG, offset_Stim, task_train=[2,3], n_components=5, MOD='EEG-EOG', MASK=False, nb_nearby_samples=None):
    significant_spatial_patterns = []
    for Subj_ID in range(len(subjects)):
        print(f"###################\nSubject {Subj_ID + 1} / {len(subjects)}")
        data_multitask_dict, objflow_list, saccade_list, blink_list = prepare_data_subj(Subj_ID, fs)
        if MASK:
            data_masked_dict, objflow_masked_list, _ = mask_data(data_multitask_dict, saccade_list, blink_list, objflow_list, nb_nearby_samples, MOD=MOD)
            data_masked_list = data_masked_dict[MOD]
        else:
            data_masked_list = None
            objflow_masked_list = None
        CCA = algo.CanonicalCorrelationAnalysis(data_multitask_dict[MOD], objflow_list, fs, L_EEG, L_Stim, offset_EEG, offset_Stim, task_train=task_train, leave_out=1, n_components=n_components, EEG_masked=data_masked_list, Stim_masked=objflow_masked_list)
        _, corr_test_fold, sig_corr_pool, forward_model_fold = CCA.cross_val(PERMU_TEST=True)
        is_significant = corr_test_fold > sig_corr_pool 
        for fold_idx in range(len(corr_test_fold)):
            for task_idx in range(3):
                for comp_idx in range(n_components):
                    # Check if this specific component, in this fold, for this task was significant
                    if is_significant[fold_idx, comp_idx, task_idx]:
                        # Extract the model (assuming forward_model_fold has shape: channels x components)
                        sig_model = forward_model_fold[fold_idx][:, comp_idx, task_idx]
                        # Store it alongside its metadata
                        significant_spatial_patterns.append({
                            'Subject_ID': Subj_ID + 1,
                            'fold': fold_idx,
                            'task': task_idx,
                            'component': comp_idx,
                            'correlation': corr_test_fold[fold_idx, comp_idx, task_idx],
                            'forward_model': sig_model
                        })

    # save as pickle file
    with open(f"tables/{MOD}/forward_models_cca.pickle", 'wb') as f:
        pickle.dump(significant_spatial_patterns, f)
    return significant_spatial_patterns

In [ ]:
def calculate_spatial_correlation(map_A, map_B):
    """Calculates the Pearson correlation coefficient between two 1D spatial maps."""
    A = map_A.reshape(-1)
    B = map_B.reshape(-1)
    return np.corrcoef(A, B)[0, 1]

def grand_average_spatial_matching(significant_patterns):
    """
    Creates a Grand Average by matching components across folds and subjects 
    based on their spatial similarity to a master template.
    """
    grand_averaged_models = {}
    unique_tasks = set(entry['task'] for entry in significant_patterns)
    
    for task_id in unique_tasks:
        # Isolate all significant components for this task
        task_data = [entry for entry in significant_patterns if entry['task'] == task_id]
        if not task_data:
            continue
        # DEFINE THE TEMPLATE
        # Find the single best component across all subjects and folds based on task correlation
        best_overall_entry = max(task_data, key=lambda x: x['correlation'])
        template_model = best_overall_entry['forward_model']
        
        aligned_matched_models = []
        
        # SEARCH AND MATCH
        # Group the data by Subject and Fold so we can search within each specific instance
        unique_subjects = set(entry['Subject_ID'] for entry in task_data)
        
        for subj_id in unique_subjects:
            subj_folds = set(entry['fold'] for entry in task_data if entry['Subject_ID'] == subj_id)
            
            for fold_id in subj_folds:
                # Get all components for this specific subject and fold
                candidates = [entry for entry in task_data 
                              if entry['Subject_ID'] == subj_id and entry['fold'] == fold_id]
                
                best_match_model = None
                highest_abs_similarity = -1
                
                # Test every candidate component against the Template
                for candidate in candidates:
                    spatial_sim = calculate_spatial_correlation(template_model, candidate['forward_model'])
                    
                    # We look for the highest ABSOLUTE similarity, because a perfect match 
                    # might just be inverted (spatial_sim = -0.99)
                    if abs(spatial_sim) > highest_abs_similarity:
                        highest_abs_similarity = abs(spatial_sim)
                        best_match_model = candidate['forward_model']
                
                # ALIGN AND COLLECT
                # Now that we found the spatial twin, check its sign against the template
                if np.dot(template_model.reshape(-1), best_match_model.reshape(-1)) < 0:
                    aligned_matched_models.append(-best_match_model)
                else:
                    aligned_matched_models.append(best_match_model)
                    
        # AVERAGE
        grand_avg = np.mean(aligned_matched_models, axis=0)
        grand_averaged_models[task_id] = grand_avg
        
        print(f"Task {task_id}: Grand Averaged {len(aligned_matched_models)} spatially matched components.")
        
    return grand_averaged_models

In [ ]:
significant_spatial_patterns = topoplot_cca(fs, L_EEG, L_Stim, offset_EEG, offset_Stim, task_train=task_train, n_components=n_components, MOD=MOD, MASK=MASK, nb_nearby_samples=nb_nearby_samples)

In [ ]:
with open(f"tables/{MOD}/forward_models_cca.pickle", 'rb') as f:
    fm_dict = pickle.load(f)
grand_averaged_models = grand_average_spatial_matching(fm_dict)
utils.plot_three_tasks(grand_averaged_models, file_name="Tasks_Grand_Average.pdf")

## Group analysis

In [ ]:
L_gcca = 5
offset_gcca = 2

MOD = 'EEG-EOG'
task_train = [1,2,3]
n_components = 5 if (MOD != 'GAZE_V' and MOD != 'GAZE') else 3
range_into_account = 3
nb_comp_into_account = 2

In [ ]:
def data_reg_feats(REGFEATS=False, SYNC=True):
    data_multitask_dict, objflow_list, _, _ = prepare_data_subj(0, fs)
    data_list = [np.expand_dims(data, axis=2) for data in data_multitask_dict[MOD]]
    for Subj_ID in range(1, len(subjects)):
        data_multitask_dict, _, _, _ = prepare_data_subj(Subj_ID, fs)
        if not SYNC:
            data_multitask_dict[MOD] = np.random.permutation(data_multitask_dict[MOD])
        data_list = [np.concatenate((data, np.expand_dims(newdata, axis=2)), axis=2) for data, newdata in zip(data_list, data_multitask_dict[MOD])]
    if REGFEATS:
        data_list = [utils.regress_out_4D(data, of) for data, of in zip(data_list, objflow_list)]
    return data_list

def data_and_masked_data(mask_videos, REGFEATS=False, SYNC=True):
    data_list = data_reg_feats(REGFEATS=REGFEATS, SYNC=SYNC)
    data_list_masked = copy.deepcopy(data_list)
    for mask, data in zip(mask_videos, data_list_masked):
            data_dim, nb_subjs = data.shape[1], data.shape[2]
            mask = np.repeat(mask[:,np.newaxis,:], data_dim, axis=1)
            mask = np.repeat(mask[:,:,np.newaxis,:], nb_subjs, axis=2)
            data[~mask] = np.nan
    return data_list, data_list_masked

In [ ]:
event_type = 'all'

mask_videos = utils.create_event_masks(RANDOMIZE=False, event_type=event_type)
data_list, data_list_masked = data_and_masked_data(mask_videos)

# Use this for significance level
GCCA = algo.GeneralizedCCA(data_list, fs, L_gcca, offset_gcca, task_train=task_train, leave_out=1, n_components=5, regularization='lwcov', message=True, signifi_level=True, EEG_list_masked=data_list_masked)
corr_train_folds, corr_test_folds, sig_corr_folds, sig_corr_pool, _, _, _ = GCCA.cross_val(CORRCA=True, sig_components=2)
forward_models_cross = GCCA.forward_model()

# with open(f'tables/EEG-EOG/isc_folds_train_{task_train}_event{event_type}_BT.pkl', 'wb') as f:
#     pickle.dump(corr_test_folds, f)
# with open(f'tables/EEG-EOG/forward_models_{event_type}_new.pickle', 'wb') as f:
#     pickle.dump(forward_models_cross, f)

In [ ]:
event_type = 'all'

mask_videos = utils.create_event_masks(RANDOMIZE=False, event_type=event_type)
data_list, data_list_masked = data_and_masked_data(mask_videos)

GCCA = algo.GeneralizedCCA(data_list, fs, L_gcca, offset_gcca, task_train=task_train, leave_out=1, n_components=5, regularization='lwcov', message=True, signifi_level=False, EEG_list_masked=data_list_masked)
corr_test_folds, start_points, _ = GCCA.cross_val_trials(BOOTSTRAP=True, trial_len=45, given_start_points=None, BTfactor=2, CORRCA=True)
forward_models_circle = GCCA.forward_model()

with open(f'tables/EEG-EOG/isc_folds_train_{task_train}_event{event_type}_BT.pkl', 'wb') as f:
    pickle.dump(corr_test_folds, f)

with open(f'tables/EEG-EOG/forward_models_{event_type}_new.pickle', 'wb') as f:
    pickle.dump(forward_models_circle, f)

### Topoplot

In [ ]:
def get_consistent_maps(forward_models):
    """
    Ensures that the maximum absolute peak of every task's forward model is positive.
    
    Parameters:
    - forward_models: List of numpy arrays, each representing a forward model for a task.
    
    Returns:
    - maps_consistent: List of numpy arrays with consistent sign for the maximum absolute peak.
    """
    maps_consistent = []
    for fm in forward_models:
        # Find the index of the largest weight (ignoring sign)
        task_map = fm.reshape(-1)  # Flatten the forward model to 1D
        peak_idx = np.argmax(np.abs(task_map))
        
        # If the actual value at that peak is negative, flip the whole map
        if task_map[peak_idx] < 0:
            fm = -fm  # Flip the sign of the entire forward model
        maps_consistent.append(fm)
    
    return maps_consistent


In [ ]:
# load corr_test_folds and forward_models_cross
with open(f'tables/EEG-EOG/isc_folds_train_{task_train}_eventall_BT.pkl', 'rb') as f:
    corr_test_folds = pickle.load(f)
with open(f'tables/EEG-EOG/forward_models_all_new.pickle', 'rb') as f:
    forward_models = pickle.load(f)
comp = 0
threshold = 0
corr_task_1 = [np.mean(fold, axis=0)[comp,0] for fold in corr_test_folds]
fm_task_1_refined = [fm[:,:,0] for fm, corr in zip(forward_models, corr_task_1) if corr > threshold]

corr_task_2 = [np.mean(fold, axis=0)[comp,1] for fold in corr_test_folds]
corr_task_3 = [np.mean(fold, axis=0)[comp,2] for fold in corr_test_folds]
fm_task_2_refined = [fm[:,:,1] for fm, corr in zip(forward_models, corr_task_2) if corr > threshold]
fm_task_3_refined = [fm[:,:,2] for fm, corr in zip(forward_models, corr_task_3) if corr > threshold]

fm_task_1_consistent = get_consistent_maps(fm_task_1_refined)
fm_task_2_consistent = get_consistent_maps(fm_task_2_refined)
fm_task_3_consistent = get_consistent_maps(fm_task_3_refined)

fm_task_1 = np.mean(fm_task_1_consistent, axis=0)
fm_task_2 = np.mean(fm_task_2_consistent, axis=0)
fm_task_3 = np.mean(fm_task_3_consistent, axis=0)


In [ ]:
utils.plot_three_tasks([fm_task_1[:, comp], fm_task_2[:, comp], fm_task_3[:, comp]], file_name="Group_topo.pdf")